## Models Like BERT or LLaMA are very large models, so its important to use quantization techniques to reduce model size , time, cost and resources. By reducing size of model the accuraccy can still be very high while reducing its size alot

## There are actually 2 Methodes 

### Methode1: Floating-point downcasting

#### `FP16 downcasting`
converts model weights from 32-bit floating-point values to 16-bit floating-point values. This halves the storage required for the converted weights. However, FP16 has lower precision and a smaller numerical range than FP32, which may affect model outputs or cause numerical overflow.

#### `BF16 Downcasting`
BF16 also uses 16 bits per value, so it provides the same weight-storage reduction as FP16. It preserves the exponent range of FP32, allowing it to represent much larger and smaller numbers than FP16. However, it provides fewer precision bits than FP16, so values are rounded more coarsely.

In [1]:
from transformers import BlipForConditionalGeneration

In [2]:
model_name = "Salesforce/blip-image-captioning-base"

In [3]:
model= BlipForConditionalGeneration.from_pretrained(model_name)

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [4]:
fp32_mem_footprint = model.get_memory_footprint()

print("Footprint of the fp32 model in bytes: ",
      fp32_mem_footprint)
print("Footprint of the fp32 model in MBs: ", 
      fp32_mem_footprint/1e+6)

Footprint of the fp32 model in bytes:  895890672
Footprint of the fp32 model in MBs:  895.890672


- Load the same model in `bfloat16`.

In [8]:
import torch 
model_bf16 = BlipForConditionalGeneration.from_pretrained(
                                               model_name,
                               torch_dtype=torch.bfloat16
)

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [9]:
bf16_mem_footprint = model_bf16.get_memory_footprint()

In [10]:
# Get the relative difference
relative_diff = bf16_mem_footprint / fp32_mem_footprint

print("Footprint of the bf16 model in MBs: ", 
      bf16_mem_footprint/1e+6)
print(f"Relative diff: {relative_diff}")

Footprint of the bf16 model in MBs:  447.947384
Relative diff: 0.5000022859932177


- exactly half size
- accuracy will be still very high

### Methode2: Integer quantization

#### `INT8 Quantization`

INT8 quantization maps floating-point values to 8-bit integers, typically using a scale and a zero point. Compared with FP32, the converted weights theoretically require one-quarter of the original storage(So instead of 4-bit per Value its just 1 Bit). This can reduce memory usage and improve inference speed on supported hardware, although prediction quality should be evaluated after quantization.

#### `INT4 Quantization`

INT4 quantization represents weights using 4-bit integers, providing stronger compression than INT8. The converted weights theoretically require one-eighth of their FP32 storage. This helps large language models fit into limited memory, but the reduced precision can introduce larger approximation errors, making the choice of quantization method important.